# TMDB Keyword Lexicon

One row per unique TMDB keyword phrase, joined to all three NRC lexicons.

**26 columns:**
- `keyword`, `n_movies`
- `emolex_found`, `intensity_found`, `vad_found`
- `emolex_anger` … `emolex_negative` (binary union across tokens)
- `intensity_anger` … `intensity_disgust` (mean across matched tokens)
- `vad_valence`, `vad_arousal`, `vad_dominance` (MWE-first, then token mean)

EmoLex uses **union** across tokens: if any token has `fear=1`, the keyword gets `emolex_fear=1`.  
Intensity and VAD use **mean** across matched tokens.  
VAD checks the full phrase as an MWE first before tokenizing.

**Known limitation:** token-level averaging on a bipolar scale is lossy for phrases not in the  
VAD MWE list. See: Mohammad et al., [SCL](https://www.saifmohammad.com/WebPages/SCL.html).

In [1]:
import pickle
import re
from collections import Counter
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

EMOTIONS   = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']
EMOLEX_COLS = EMOTIONS + ['positive', 'negative']
VAD_DIMS   = ['valence', 'arousal', 'dominance']

_lemmatizer = WordNetLemmatizer()

def _tokenize(phrase: str) -> list:
    return [t for t in re.split(r'[\s\-_/]+', phrase.lower()) if t.isalpha()]

def _lemma_candidates(tok: str) -> list:
    seen, out = set(), []
    for pos in ('n', 'v', 'a'):
        lemma = _lemmatizer.lemmatize(tok, pos=pos)
        if lemma not in seen:
            seen.add(lemma)
            out.append(lemma)
    if tok not in seen:
        out.append(tok)
    return out


## Load lexicons

In [2]:
# ── EmoLex ────────────────────────────────────────────────────────────────
NRC_EMOLEX = Path('data/NRC-emotion-lexicon-wordlevel-alphabetized-v0.92.txt')
nrc_long = pd.read_csv(NRC_EMOLEX, sep='\t', header=None,
                       names=['word', 'emotion', 'value'], skiprows=46)
emolex_wide = nrc_long.pivot(index='word', columns='emotion', values='value').reset_index()
emolex_lookup = {row['word']: {e: int(row[e]) for e in EMOLEX_COLS}
                 for _, row in emolex_wide.fillna(0).iterrows()}
print(f'EmoLex:    {len(emolex_lookup):,} words')

# ── Intensity ──────────────────────────────────────────────────────────────
INTENSITY_FILE = Path('data/NRC-Emotion-Intensity-Lexicon/NRC-Emotion-Intensity-Lexicon-v1.txt')
nrc_int_long = pd.read_csv(INTENSITY_FILE, sep='\t', header=None,
                           names=['word', 'emotion', 'intensity'])
nrc_int_wide = (nrc_int_long.pivot(index='word', columns='emotion', values='intensity')
                .fillna(0).reset_index())
intensity_lookup = {row['word']: {e: row[e] for e in EMOTIONS}
                    for _, row in nrc_int_wide.iterrows()}
print(f'Intensity: {len(intensity_lookup):,} words')

# ── VAD v2.1 ───────────────────────────────────────────────────────────────
VAD_FILE = Path('data/NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt')
nrc_vad = pd.read_csv(VAD_FILE, sep='\t')
vad_lookup = {row['term']: {d: row[d] for d in VAD_DIMS} for _, row in nrc_vad.iterrows()}
print(f'VAD v2.1:  {len(vad_lookup):,} entries (unigrams + MWEs)')


EmoLex:    14,150 words
Intensity: 5,891 words


VAD v2.1:  54,801 entries (unigrams + MWEs)


## Extract unique TMDB keywords

In [ ]:
import kagglehub

_ds_path = kagglehub.dataset_download('asaniczka/tmdb-movies-dataset-2023-930k-movies')
TMDB_FILE = next(Path(_ds_path).glob('**/*.csv'))
tmdb = pd.read_csv(TMDB_FILE, usecols=['keywords'], low_memory=False)

kw_counter = Counter()
for cell in tmdb['keywords'].dropna():
    for kw in str(cell).split(', '):
        kw = kw.strip()
        if kw:
            kw_counter[kw] += 1

print(f'Unique keywords: {len(kw_counter):,}')
print(f'Total keyword occurrences: {sum(kw_counter.values()):,}')


## Score each keyword against all three lexicons

In [4]:
def score_keyword(keyword):
    tokens = _tokenize(keyword)
    row = {'keyword': keyword, 'n_movies': kw_counter[keyword]}

    # ── EmoLex: union across tokens ───────────────────────────────────────
    emo_union = {e: 0 for e in EMOLEX_COLS}
    emolex_hit = False
    for tok in tokens:
        for lemma in _lemma_candidates(tok):
            if lemma in emolex_lookup:
                emolex_hit = True
                for e in EMOLEX_COLS:
                    emo_union[e] = max(emo_union[e], emolex_lookup[lemma][e])
                break  # first matching lemma per token
    row['emolex_found'] = emolex_hit
    for e in EMOLEX_COLS:
        row[f'emolex_{e}'] = emo_union[e] if emolex_hit else None

    # ── Intensity: mean across matched tokens ─────────────────────────────
    int_rows = []
    for tok in tokens:
        for lemma in _lemma_candidates(tok):
            if lemma in intensity_lookup:
                int_rows.append(intensity_lookup[lemma])
                break
    row['intensity_found'] = bool(int_rows)
    for e in EMOTIONS:
        row[f'intensity_{e}'] = float(np.mean([r[e] for r in int_rows])) if int_rows else None

    # ── VAD: MWE-first, then token mean ──────────────────────────────────
    if keyword in vad_lookup:
        vad_rows = [vad_lookup[keyword]]
    else:
        vad_rows = []
        for tok in tokens:
            for lemma in _lemma_candidates(tok):
                if lemma in vad_lookup:
                    vad_rows.append(vad_lookup[lemma])
                    break
    row['vad_found'] = bool(vad_rows)
    for d in VAD_DIMS:
        row[f'vad_{d}'] = float(np.mean([r[d] for r in vad_rows])) if vad_rows else None

    return row

print('Scoring keywords...')
records = [score_keyword(kw) for kw in kw_counter]
kw_df = pd.DataFrame(records)

# Column order
col_order = (
    ['keyword', 'n_movies',
     'emolex_found', 'intensity_found', 'vad_found']
    + [f'emolex_{e}' for e in EMOLEX_COLS]
    + [f'intensity_{e}' for e in EMOTIONS]
    + [f'vad_{d}' for d in VAD_DIMS]
)
kw_df = kw_df[col_order].sort_values('n_movies', ascending=False).reset_index(drop=True)

print(f'Done. Shape: {kw_df.shape}')
print(f'\nCoverage:')
print(f'  emolex_found:    {kw_df["emolex_found"].sum():,} / {len(kw_df):,} ({kw_df["emolex_found"].mean()*100:.1f}%)')
print(f'  intensity_found: {kw_df["intensity_found"].sum():,} / {len(kw_df):,} ({kw_df["intensity_found"].mean()*100:.1f}%)')
print(f'  vad_found:       {kw_df["vad_found"].sum():,} / {len(kw_df):,} ({kw_df["vad_found"].mean()*100:.1f}%)')
kw_df.head(10)


Scoring keywords...


Done. Shape: (67916, 26)

Coverage:
  emolex_found:    37,678 / 67,916 (55.5%)
  intensity_found: 21,405 / 67,916 (31.5%)
  vad_found:       44,606 / 67,916 (65.7%)


,keyword,n_movies,emolex_found,intensity_found,vad_found,emolex_anger,emolex_anticipation,emolex_disgust,emolex_fear,emolex_joy,...,intensity_anticipation,intensity_disgust,intensity_fear,intensity_joy,intensity_sadness,intensity_surprise,intensity_trust,vad_valence,vad_arousal,vad_dominance
0,short film,29323,True,False,True,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00700,-0.245,-0.19700
1,woman director,16004,True,True,True,0.0,0.0,0.0,0.0,0.0,...,0.000,0.000,0.000,0.000,0.000,0.000,0.445,0.37600,0.160,0.82400
2,gay pornography,15466,True,True,True,0.0,0.0,1.0,0.0,0.0,...,0.000,0.586,0.000,0.000,0.000,0.000,0.000,-0.35400,0.842,-0.03800
3,based on novel or book,6555,True,True,True,0.0,0.0,0.0,0.0,0.0,...,0.000,0.000,0.000,0.000,0.000,0.000,0.430,0.27975,-0.219,0.14375
4,anal sex,5938,True,True,True,0.0,1.0,0.0,0.0,1.0,...,0.695,0.000,0.000,0.622,0.000,0.000,0.375,-0.53400,0.318,-0.27600
5,concert,5220,True,False,True,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.87600,0.734,0.43600
6,lgbt,5218,False,False,False,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,murder,4932,True,True,True,1.0,0.0,1.0,1.0,0.0,...,0.000,0.839,0.906,0.000,0.828,0.617,0.000,-0.87800,0.884,0.23000
8,compilation,4932,True,False,True,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.06000,0.024,0.21200
9,biography,4079,True,False,True,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.39600,-0.542,0.43600


## Export

In [5]:
OUT_DIR = Path('output')
OUT_DIR.mkdir(exist_ok=True)
out_path = OUT_DIR / 'tmdb_keyword_lexicon.csv'
kw_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)')


Saved: output/tmdb_keyword_lexicon.csv  (6.5 MB)
